In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, DoubleType, LongType, StringType, StructField, StructType,
)

if "batch_id" not in dbutils.widgets.getAll():
    dbutils.widgets.text("batch_id", "full-sample-20261010-01", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze_batch_df = (
    spark.table("workspace.tmdb_lakehouse.bronze_movie_records")
    .filter(F.col("batch_id") == batch_id)
)
bronze_count = bronze_batch_df.count()
if bronze_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

review_movie_schema = StructType([
    StructField("id", LongType(), True),
    StructField("release_date", StringType(), True),
    StructField("reviews", StructType([StructField("results", ArrayType(StructType([
        StructField("id", StringType(), True),
        StructField("author", StringType(), True),
        StructField("author_details", StructType([
            StructField("username", StringType(), True),
            StructField("name", StringType(), True),
            StructField("avatar_path", StringType(), True),
            StructField("rating", DoubleType(), True),
        ]), True),
        StructField("content", StringType(), True),
        StructField("created_at", StringType(), True),
        StructField("updated_at", StringType(), True),
        StructField("url", StringType(), True),
    ])), True)]), True),
])

parsed_reviews_df = (
    bronze_batch_df
    .select(
        "movie_id", "batch_id", "load_type", "source_file", "source_extracted_at",
        F.from_json("raw_payload", review_movie_schema).alias("movie"),
    )
    .select(
        "movie_id", "batch_id", "load_type",
        "source_file", "source_extracted_at", "movie.*",
    )
)

bad_ids = parsed_reviews_df.filter(
    F.col("id").isNull() | (F.col("id") != F.col("movie_id"))
).count()
if bad_ids:
    raise ValueError(f"{bad_ids} Bronze records have a missing or mismatched movie ID")

reviews_present_df = parsed_reviews_df.filter(F.col("reviews.results").isNotNull())
review_total = (
    parsed_reviews_df
    .select(F.sum(F.coalesce(F.size("reviews.results"), F.lit(0))).alias("n"))
    .first()["n"] or 0
)

print(f"Bronze rows in selected batch: {bronze_count}")
print(f"Parsed movie rows: {parsed_reviews_df.count()}")
print(f"Movies with a present reviews collection: {reviews_present_df.count()}")
print(f"Movies with an empty reviews collection: {reviews_present_df.filter(F.size('reviews.results') == 0).count()}")
print(f"Movies with a missing or unparsed reviews collection: {parsed_reviews_df.filter(F.col('reviews.results').isNull()).count()}")
print(f"Review rows in selected batch: {review_total}")

In [0]:
review_hash_salt = dbutils.secrets.get(
    scope="tmdb-lakehouse-secrets",
    key="reviewer-hash-salt",
)

exploded_reviews_df = (
    reviews_present_df
    .select(
        "movie_id", "batch_id", "source_file", "source_extracted_at", "release_date",
        F.explode("reviews.results").alias("review"),
    )
    .withColumn(
        "reviewer_identity",
        F.coalesce(
            F.when(
                F.length(F.trim(F.col("review.author_details.username"))) > 0,
                F.trim(F.col("review.author_details.username")),
            ),
            F.when(
                F.length(F.trim(F.col("review.author"))) > 0,
                F.trim(F.col("review.author")),
            ),
        ),
    )
    .withColumn(
        "reviewer_hash_id",
        F.when(
            F.col("reviewer_identity").isNotNull(),
            F.sha2(
                F.concat(
                    F.lit(review_hash_salt), F.lit(":"),
                    F.lower(F.col("reviewer_identity")),
                ),
                256,
            ),
        ),
    )
    .withColumn(
        "review_content_scrubbed",
        F.regexp_replace(
            F.regexp_replace(
                F.col("review.content"),
                r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
                "[EMAIL]",
            ),
            r"(?i)\b(?:https?://|www\.)\S+",
            "[URL]",
        ),
    )
    .withColumn("review_created_at", F.to_timestamp(F.col("review.created_at")))
    .withColumn("review_updated_at", F.to_timestamp(F.col("review.updated_at")))
    .withColumn("release_date_parsed", F.to_date(F.col("release_date"), "yyyy-MM-dd"))
)

silver_reviews_source_df = exploded_reviews_df.select(
    "movie_id",
    F.col("review.id").alias("review_id"),
    "reviewer_hash_id",
    F.col("review_content_scrubbed").alias("content"),
    F.col("review_created_at").alias("created_at"),
    F.col("review_updated_at").alias("updated_at"),
    F.col("review.author_details.rating").cast("double").alias("rating"),
    F.floor(
        (
            F.unix_timestamp("review_created_at")
            - F.unix_timestamp(F.col("release_date_parsed").cast("timestamp"))
        ) / F.lit(7 * 24 * 60 * 60)
    ).cast("int").alias("weeks_since_release"),
    F.col("batch_id").alias("source_batch_id"),
    "source_file", "source_extracted_at",
)

review_hash_columns = [
    "movie_id", "review_id", "reviewer_hash_id", "content",
    "created_at", "updated_at", "rating", "weeks_since_release",
]
silver_reviews_source_df = (
    silver_reviews_source_df
    .withColumn("row_hash", F.sha2(F.to_json(F.struct(*review_hash_columns)), 256))
    .withColumn("load_timestamp", F.current_timestamp())
)

invalid_review_keys = silver_reviews_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("review_id").isNull()
    | (F.length(F.trim(F.col("review_id"))) == 0)
).count()
invalid_created_timestamps = exploded_reviews_df.filter(
    F.col("review.created_at").isNull() | F.col("review_created_at").isNull()
).count()
conflicting_review_keys = (
    silver_reviews_source_df.groupBy("movie_id", "review_id")
    .agg(F.countDistinct("row_hash").alias("versions"))
    .filter(F.col("versions") > 1)
    .count()
)
duplicate_review_rows = (
    silver_reviews_source_df.count()
    - silver_reviews_source_df.select("movie_id", "review_id").dropDuplicates().count()
)

if invalid_review_keys:
    raise ValueError(
        f"Found {invalid_review_keys} review rows missing a valid movie_id or review_id"
    )
if invalid_created_timestamps:
    raise ValueError(
        f"Found {invalid_created_timestamps} reviews with missing or invalid created_at"
    )
if conflicting_review_keys:
    raise ValueError(f"Found {conflicting_review_keys} review keys with conflicting content")

silver_reviews_source_df = silver_reviews_source_df.dropDuplicates(["movie_id", "review_id"])

print("Review rows prepared:", silver_reviews_source_df.count())
print("Invalid review keys:", invalid_review_keys)
print("Missing or invalid review created_at:", invalid_created_timestamps)
print("Conflicting movie-review keys:", conflicting_review_keys)
print("Identical duplicate review rows collapsed:", duplicate_review_rows)
print(
    "Reviews with no reviewer identity (hash is NULL):",
    silver_reviews_source_df.filter(F.col("reviewer_hash_id").isNull()).count(),
)
print(
    "Reviews with no parseable release date (weeks_since_release is NULL):",
    silver_reviews_source_df.filter(F.col("weeks_since_release").isNull()).count(),
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movie_reviews (
    movie_id BIGINT, review_id STRING, reviewer_hash_id STRING, content STRING,
    created_at TIMESTAMP, updated_at TIMESTAMP, rating DOUBLE,
    weeks_since_release INT, source_batch_id STRING, source_file STRING,
    source_extracted_at TIMESTAMP, row_hash STRING, load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movie_reviews").show(truncate=False)

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql.types import TimestampType

reviews_table = "workspace.tmdb_lakehouse.silver_movie_reviews"
logs_table = "workspace.tmdb_lakehouse.pipeline_execution_logs"

invalid_lineage_count = silver_reviews_source_df.filter(
    F.col("source_batch_id").isNull() | F.col("source_extracted_at").isNull()
).count()
if invalid_lineage_count:
    raise ValueError(
        f"{invalid_lineage_count} review rows have missing batch or extraction timestamp"
    )

review_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

review_run_id = str(uuid4())
review_run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
review_rows_inserted = 0
review_rows_updated = 0
review_rows_quarantined = 0


def write_review_run_log(status, error_details=None):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    load_type = parsed_reviews_df.select("load_type").first()["load_type"]
    row = [(
        review_run_id, "Bronze-to-Silver", load_type,
        f"batch_id={batch_id}; table=silver_movie_reviews",
        batch_id, review_run_started_at, finished_at, status,
        review_rows_inserted, review_rows_updated, review_rows_quarantined,
        error_details, finished_at,
    )]
    (
        spark.createDataFrame(row, schema=review_log_schema)
        .write.format("delta").mode("append").saveAsTable(logs_table)
    )


try:
    existing_reviews_df = spark.table(reviews_table)

    # New reviews can arrive from backfills, but an older snapshot cannot
    # overwrite an existing review. Same-batch retries remain eligible.
    eligible_review_source_df = (
        silver_reviews_source_df.alias("incoming")
        .join(
            existing_reviews_df.select(
                "movie_id", "review_id",
                F.col("source_extracted_at").alias("stored_extracted_at"),
                F.col("source_batch_id").alias("stored_batch_id"),
            ).alias("stored"),
            on=["movie_id", "review_id"], how="left",
        )
        .filter(
            F.col("stored_batch_id").isNull()
            | (F.col("incoming.source_extracted_at") > F.col("stored_extracted_at"))
            | (
                (F.col("incoming.source_extracted_at") == F.col("stored_extracted_at"))
                & (F.col("incoming.source_batch_id") == F.col("stored_batch_id"))
            )
        )
        .select("incoming.*")
    )

    review_rows_inserted = (
        silver_reviews_source_df.select("movie_id", "review_id").distinct()
        .join(
            existing_reviews_df.select("movie_id", "review_id"),
            on=["movie_id", "review_id"], how="left_anti",
        )
        .count()
    )

    review_rows_updated = (
        eligible_review_source_df.alias("incoming")
        .join(
            existing_reviews_df.alias("stored"),
            on=["movie_id", "review_id"], how="inner",
        )
        .filter(
            ~F.col("incoming.row_hash").eqNullSafe(F.col("stored.row_hash"))
            | ~F.col("incoming.source_extracted_at").eqNullSafe(F.col("stored.source_extracted_at"))
            | ~F.col("incoming.source_batch_id").eqNullSafe(F.col("stored.source_batch_id"))
            | ~F.col("incoming.source_file").eqNullSafe(F.col("stored.source_file"))
        )
        .count()
    )

    update_columns = [
        column for column in silver_reviews_source_df.columns
        if column not in ("movie_id", "review_id", "load_timestamp")
    ]
    assignments = ", ".join(f"target.{c} = source.{c}" for c in update_columns) + (
        ", target.load_timestamp = CASE WHEN NOT (target.row_hash <=> source.row_hash)"
        " THEN source.load_timestamp ELSE target.load_timestamp END"
    )

    eligible_review_source_df.createOrReplaceTempView("eligible_reviews_source")
    spark.sql(f"""
        MERGE INTO {reviews_table} AS target
        USING eligible_reviews_source AS source
        ON target.movie_id = source.movie_id
           AND target.review_id = source.review_id
        WHEN MATCHED AND (
            source.source_extracted_at > target.source_extracted_at
            OR (
                source.source_extracted_at = target.source_extracted_at
                AND source.source_batch_id = target.source_batch_id
            )
        ) AND (
            NOT (target.row_hash <=> source.row_hash)
            OR NOT (target.source_extracted_at <=> source.source_extracted_at)
            OR NOT (target.source_batch_id <=> source.source_batch_id)
            OR NOT (target.source_file <=> source.source_file)
        )
        THEN UPDATE SET {assignments}
        WHEN NOT MATCHED THEN INSERT *
    """)

    write_review_run_log("SUCCESS")
    print("Review rows inserted:", review_rows_inserted)
    print("Review rows updated:", review_rows_updated)
    print("Review rows quarantined:", review_rows_quarantined)
    print("Review rows now stored:", spark.table(reviews_table).count())

except Exception as exc:
    try:
        write_review_run_log("FAILURE", str(exc)[:4000])
    except Exception as log_exc:
        raise RuntimeError(
            f"Review processing failed: {exc}; failure logging also failed: {log_exc}"
        ) from log_exc
    raise